### PII Detection MiddleWare

Implementing a robust PII Detection system allows you to identify and manage sensitive personal data within your conversations using customizable workflows. This process is essential for:

    Regulatory Compliance: Meeting strict legal standards for healthcare (HIPAA) and financial services (GLBA/GDPR).
    
    Data Sanitization: Helping customer support teams automatically redact sensitive details from chat logs and transcripts.
    
    Privacy Protection: Securing any application that processes or stores confidential user information.

### Problem Statement: Securing PII in LLM-Driven Financial Workflows

### The Challenge

As financial institutions integrate Large Language Models (LLMs) into their customer service workflows, they face a critical security-utility trade-off. While LLMs can efficiently handle account inquiries and support tickets, they often require access to sensitive user data.

Inadvertently passing Personally Identifiable Information (PII)—such as email addresses, credit card numbers, or account IDs—to third-party LLM providers (like AWS Bedrock, OpenAI, or Anthropic) poses significant risks, including:

    Regulatory Non-Compliance: Violating mandates such as GDPR, CCPA, or PCI-DSS.

    Data Leakage: The risk of sensitive data being stored in provider logs or used for future model training.

### The Requirement

    The organization needs an automated "Banking Assistant" capable of executing functional tasks (checking balances and logging CRM tickets) without allowing sensitive user identifiers to leave the local or secure environment.

### The Solution

    Implement a Middleware-First Architecture for PII redaction. By intercepting the user’s raw input before it reaches the LLM, the system:
    
    Identifies and Redacts: Uses a PIIMiddleware layer to strip out sensitive patterns (emails, credit cards) in real-time.
    
    Maintains Tool Context: Allows the agent to process functional strings (like ACC-990) to trigger backend tools while keeping the user's identity obscured from the model logic.
    
    Separates Concerns: Decouples data privacy (handled by the middleware) from reasoning and task execution (handled by the LLM).

In [2]:
import os
from langchain_aws import ChatBedrockConverse
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.tools import tool

# Initialize the Enterprise LLM

llm = ChatBedrockConverse(
    model_id="cohere.command-r-plus-v1:0",
    region_name="us-east-1"
)


In [3]:
from rich import print
print(llm.invoke("Hi"))

AIMessage(
    content='Hello! How can I help you today?',
    additional_kwargs={},
    response_metadata={
        'ResponseMetadata': {
            'RequestId': 'ee714ca8-e3a5-4e58-ab1c-5fedc55a2843',
            'HTTPStatusCode': 200,
            'HTTPHeaders': {
                'date': 'Fri, 27 Mar 2026 11:22:47 GMT',
                'content-type': 'application/json',
                'content-length': '232',
                'connection': 'keep-alive',
                'x-amzn-requestid': 'ee714ca8-e3a5-4e58-ab1c-5fedc55a2843'
            },
            'RetryAttempts': 0
        },
        'stopReason': 'end_turn',
        'metrics': {'latencyMs': [429]},
        'model_provider': 'bedrock_converse',
        'model_name': 'cohere.command-r-plus-v1:0'
    },
    id='lc_run--019d2f08-4ede-7820-a95a-5d572ec5717b-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 1,
        'output_tokens': 9,
        'total_tokens': 10,
        'input_token_details': {'cache_creation': 0, 'cache_read': 0}
    }
)

In [5]:

# Define Two Industry-Relevant Tools
@tool
def check_account_balance(account_id: str) -> str:
    """Retrieves the current balance for a specific account ID."""
    # In actual use case: This would call a secure banking API
    return f"The balance for account {account_id} is $12,450.00."

@tool
def log_support_ticket(priority: str, category: str) -> str:
    """Creates a support ticket in the internal CRM."""
    return f"Ticket created: [Priority: {priority}] [Category: {category}]."

tools = [check_account_balance, log_support_ticket]

# Create the Agent with PII Compliance
# The PIIMiddleware intercepts the stream before it reaches Bedrock
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a secure banking assistant. "
        "Sensitive data in your input has been masked for security. "
        "Process the user's request using the available tools."
    ),
    middleware=[
        # Strategy 'redact' removes the info entirely
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        # Strategy 'mask' replaces characters (e.g., ****-****-****-1234)
        PIIMiddleware("credit_card", strategy="redact", apply_to_input=True),
    ],
)

# Real-Time Execution Loop
def run_secure_session(user_query: str):
    print(f"--- Processing Request ---\nRaw Input: {user_query}\n")
    
    # The agent.invoke handles the middleware logic internally
    # It redacts the PII before the LLM 'sees' the message
    response = agent.invoke({"messages": [("user", user_query)]})
    print(response['messages'])
    # Extract the final AI message from the conversation history
    final_output = response["messages"][-1].content
    return final_output

# --- Test Case ---
raw_input = (
    "My email is vip-client@gmail.com."
    "Check my balance for account ACC-990 and log a 'High' priority ticket for billing."
)

result = run_secure_session(raw_input)

print("--- Agent Final Response ---")
print(result)

--- Processing Request ---
Raw Input: My email is vip-client@gmail.com.Check my balance for account ACC-990 and log a 'High' priority ticket 
for billing.

[
    HumanMessage(
        content="My email is [REDACTED_EMAIL] my balance for account ACC-990 and log a 'High' priority ticket for 
billing.",
        additional_kwargs={},
        response_metadata={},
        id='406bdc52-f8f7-44b4-a14b-0d5b9ff33436'
    ),
    AIMessage(
        content=[
            {
                'type': 'text',
                'text': 'I will check the balance of the account and log a support ticket for billing.'
            },
            {
                'type': 'tool_use',
                'name': 'check_account_balance',
                'input': {'account_id': 'ACC-990'},
                'id': 'tooluse_jB1yXO1PSwOZu9kvOMcyJH'
            },
            {
                'type': 'tool_use',
                'name': 'log_support_ticket',
                'input': {'category': 'billing', 'priority': 'high'},
                'id': 'tooluse_wLVEP55P6kGPQbsNL8K48z'
            }
        ],
        additional_kwargs={},
        response_metadata={
            'ResponseMetadata': {
                'RequestId': 'ed2e827e-34c2-4529-986c-1ca5b2627639',
                'HTTPStatusCode': 200,
                'HTTPHeaders': {
                    'date': 'Fri, 27 Mar 2026 11:23:28 GMT',
                    'content-type': 'application/json',
                    'content-length': '540',
                    'connection': 'keep-alive',
                    'x-amzn-requestid': 'ed2e827e-34c2-4529-986c-1ca5b2627639'
                },
                'RetryAttempts': 0
            },
            'stopReason': 'tool_use',
            'metrics': {'latencyMs': [2964]},
            'model_provider': 'bedrock_converse',
            'model_name': 'cohere.command-r-plus-v1:0'
        },
        id='lc_run--019d2f08-e3d3-7323-b3f7-1e1a802c9cf2-0',
        tool_calls=[
            {
                'name': 'check_account_balance',
                'args': {'account_id': 'ACC-990'},
                'id': 'tooluse_jB1yXO1PSwOZu9kvOMcyJH',
                'type': 'tool_call'
            },
            {
                'name': 'log_support_ticket',
                'args': {'category': 'billing', 'priority': 'high'},
                'id': 'tooluse_wLVEP55P6kGPQbsNL8K48z',
                'type': 'tool_call'
            }
        ],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 90,
            'output_tokens': 46,
            'total_tokens': 136,
            'input_token_details': {'cache_creation': 0, 'cache_read': 0}
        }
    ),
    ToolMessage(
        content='The balance for account ACC-990 is $12,450.00.',
        name='check_account_balance',
        id='a0462327-3253-492f-b3aa-88a4fc2e274d',
        tool_call_id='tooluse_jB1yXO1PSwOZu9kvOMcyJH'
    ),
    ToolMessage(
        content='Ticket created: [Priority: high] [Category: billing].',
        name='log_support_ticket',
        id='d24fe586-fb3f-4646-b01c-43b821c690e3',
        tool_call_id='tooluse_wLVEP55P6kGPQbsNL8K48z'
    ),
    AIMessage(
        content='The balance for account ACC-990 is $12,450.00. A high-priority ticket for billing has been 
created.',
        additional_kwargs={},
        response_metadata={
            'ResponseMetadata': {
                'RequestId': 'a203951c-3790-4fc8-9ae6-2d8304cdce15',
                'HTTPStatusCode': 200,
                'HTTPHeaders': {
                    'date': 'Fri, 27 Mar 2026 11:23:31 GMT',
                    'content-type': 'application/json',
                    'content-length': '304',
                    'connection': 'keep-alive',
                    'x-amzn-requestid': 'a203951c-3790-4fc8-9ae6-2d8304cdce15'
                },
                'RetryAttempts': 0
            },
            'stopReason': 'end_turn',
            'metrics': {'latencyMs': [3182]},
            'model_provider': 'bedrock_converse',
            'model_name': 'cohere.command-r-plus-v1:0'
        },
        id='lc_run--019d2f08-f03f-7420-8f5c-e2ec220bd654-0',
        tool_calls=[],
       

--- Agent Final Response ---

The balance for account ACC-990 is $12,450.00. A high-priority ticket for billing has been created.